In [1]:
# A0 — Install TF 2.20
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [2]:
# A1 — Imports and config
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc
import re
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

CFG = {
    "batch_files": 64,   # 如果内存不够就改 32
    "proxy_reduce": "max",
    "verbose": True,
}

print("TensorFlow:", tf.__version__)
print("BASE exists:", BASE.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())

TensorFlow: 2.20.0
BASE exists: True
MODEL_DIR exists: True


In [3]:
# A2 — Load competition metadata
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {
            "file_id": None,
            "site": None,
            "date": pd.NaT,
            "time_utc": None,
            "hour_utc": -1,
            "month": -1,
        }
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {
        "file_id": file_id,
        "site": site,
        "date": dt,
        "time_utc": hms,
        "hour_utc": int(hms[:2]),
        "month": int(dt.month) if pd.notna(dt) else -1,
    }

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)

meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)

Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print("full files:", len(full_files))
print("trusted windows:", len(full_truth))
print("active classes:", int((Y_FULL_TRUTH.sum(axis=0) > 0).sum()))

full files: 59
trusted windows: 708
active classes: 71


In [4]:
# A3 — Load Perch + mapping/proxy
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

taxonomy = taxonomy.copy()
taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"]

bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})
mapping = taxonomy.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup",
    how="left"
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits

proxy_map = {}
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "bc_indices": hits["bc_index"].astype(int).tolist(),
            "genus": genus,
        }

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys()
    if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])

selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

print("mapped:", int(MAPPED_MASK.sum()), "/", N_CLASSES)
print("proxy targets:", len(SELECTED_PROXY_TARGETS))

mapped: 203 / 234
proxy targets: 3


In [5]:
# A4 — Perch inference helpers
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_with_embeddings(paths, batch_files=64, verbose=True, proxy_reduce="max"):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)

    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)

        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem

            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])

            x_pos += N_WINDOWS
            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32, copy=False)
        emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            proxy_score = sub.max(axis=1) if proxy_reduce == "max" else sub.mean(axis=1)
            scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)

        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids,
        "filename": filenames,
        "site": sites,
        "hour_utc": hours,
    })
    return meta_df, scores, embeddings

In [6]:
# A5 — Fit prior tables
EPS = 1e-4

def fit_prior_tables(full_truth_df, y_true):
    df = full_truth_df.copy().reset_index(drop=True)
    global_prob = y_true.mean(axis=0).astype(np.float32)

    def group_prob(col_names):
        tmp = pd.DataFrame(y_true, columns=PRIMARY_LABELS)
        grp = pd.concat([df[col_names], tmp], axis=1)
        return grp.groupby(col_names)[PRIMARY_LABELS].mean().astype(np.float32)

    tables = {
        "global": global_prob,
        "site": group_prob(["site"]),
        "hour": group_prob(["hour_utc"]),
        "month": group_prob(["month"]),
        "site_hour": group_prob(["site", "hour_utc"]),
    }
    return tables

PRIOR_TABLES = fit_prior_tables(full_truth, Y_FULL_TRUTH)

print("Prior tables ready.")
print("site groups:", len(PRIOR_TABLES["site"]))
print("hour groups:", len(PRIOR_TABLES["hour"]))
print("month groups:", len(PRIOR_TABLES["month"]))
print("site_hour groups:", len(PRIOR_TABLES["site_hour"]))

Prior tables ready.
site groups: 8
hour groups: 13
month groups: 7
site_hour groups: 23


In [7]:
# A6 — Build probe artifact from trusted full files
full_train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]

meta_full_probe, scores_full_raw_probe, emb_full_probe = infer_perch_with_embeddings(
    full_train_paths,
    batch_files=CFG["batch_files"],
    verbose=CFG["verbose"],
    proxy_reduce=CFG["proxy_reduce"],
)

assert len(meta_full_probe) == len(full_truth)

scaler_probe = StandardScaler()
X_full_scaled = scaler_probe.fit_transform(emb_full_probe)

PCA_DIM = 128
pca_probe = PCA(n_components=PCA_DIM, random_state=42)
X_full_pca = pca_probe.fit_transform(X_full_scaled)

active_class_idx = np.where(Y_FULL_TRUTH.sum(axis=0) > 0)[0].astype(np.int32)

coef_mat = np.zeros((len(active_class_idx), PCA_DIM), dtype=np.float32)
intercept_vec = np.zeros(len(active_class_idx), dtype=np.float32)

for k, j in enumerate(tqdm(active_class_idx, desc="Training simple probes")):
    yj = Y_FULL_TRUTH[:, j]
    clf = LogisticRegression(
        C=1.0,
        max_iter=300,
        solver="liblinear",
        class_weight="balanced",
        random_state=42,
    )
    clf.fit(X_full_pca, yj)
    coef_mat[k] = clf.coef_[0].astype(np.float32)
    intercept_vec[k] = np.float32(clf.intercept_[0])

print("emb_full_probe:", emb_full_probe.shape)
print("X_full_pca:", X_full_pca.shape)
print("trained probe classes:", len(active_class_idx))

Perch batches:   0%|          | 0/1 [00:00<?, ?it/s]

I0000 00:00:1776603180.508568      67 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Training simple probes:   0%|          | 0/71 [00:00<?, ?it/s]

emb_full_probe: (708, 1536)
X_full_pca: (708, 128)
trained probe classes: 71


In [8]:
# A6.5 — Build temporal-lite file-level features from 12-window raw logits

from sklearn.linear_model import LogisticRegression

# scores_full_raw_probe: shape (708, 234)
# full files: 59
# each file has 12 windows
n_full_files = len(full_files)

assert scores_full_raw_probe.shape[0] == n_full_files * N_WINDOWS

# reshape to (n_files, 12, 234)
scores_full_seq = scores_full_raw_probe.reshape(n_full_files, N_WINDOWS, N_CLASSES)

# build file-level labels: if any of 12 windows is positive, file label = 1
Y_FILE_TRUTH = Y_FULL_TRUTH.reshape(n_full_files, N_WINDOWS, N_CLASSES).max(axis=1).astype(np.uint8)

# temporal-lite per-class features:
# mean, max, std, last-first  => 4 features per class
feat_mean = scores_full_seq.mean(axis=1)                         # (n_files, 234)
feat_max = scores_full_seq.max(axis=1)                          # (n_files, 234)
feat_std = scores_full_seq.std(axis=1)                          # (n_files, 234)
feat_delta = scores_full_seq[:, -1, :] - scores_full_seq[:, 0, :]  # (n_files, 234)

print("scores_full_seq:", scores_full_seq.shape)
print("Y_FILE_TRUTH:", Y_FILE_TRUTH.shape)
print("feat_mean:", feat_mean.shape)

scores_full_seq: (59, 12, 234)
Y_FILE_TRUTH: (59, 234)
feat_mean: (59, 234)


In [9]:
# A6.6 — Train temporal-lite logistic heads (4 features per class)

temp_active_class_idx = np.where(Y_FILE_TRUTH.sum(axis=0) > 0)[0].astype(np.int32)

# For each active class j, train a 4-feature logistic regression:
# X_j = [mean_j, max_j, std_j, delta_j]
temp_coef_mat = np.zeros((len(temp_active_class_idx), 4), dtype=np.float32)
temp_intercept_vec = np.zeros(len(temp_active_class_idx), dtype=np.float32)

for k, j in enumerate(tqdm(temp_active_class_idx, desc="Training temporal-lite heads")):
    yj = Y_FILE_TRUTH[:, j]

    Xj = np.stack([
        feat_mean[:, j],
        feat_max[:, j],
        feat_std[:, j],
        feat_delta[:, j],
    ], axis=1).astype(np.float32)

    clf = LogisticRegression(
        C=1.0,
        max_iter=300,
        solver="liblinear",
        class_weight="balanced",
        random_state=42,
    )
    clf.fit(Xj, yj)

    temp_coef_mat[k] = clf.coef_[0].astype(np.float32)
    temp_intercept_vec[k] = np.float32(clf.intercept_[0])

print("temp_active_class_idx:", temp_active_class_idx.shape)
print("temp_coef_mat:", temp_coef_mat.shape)
print("temp_intercept_vec:", temp_intercept_vec.shape)

Training temporal-lite heads:   0%|          | 0/71 [00:00<?, ?it/s]

temp_active_class_idx: (71,)
temp_coef_mat: (71, 4)
temp_intercept_vec: (71,)


In [10]:
# A7 — Serialize all artifacts, including temporal-lite heads
def serialize_prior_table(df):
    if isinstance(df.index, pd.MultiIndex):
        keys = list(df.index)
    else:
        keys = list(df.index.tolist())
    vals = df.to_numpy(dtype=np.float32)
    return keys, vals

site_keys, site_vals = serialize_prior_table(PRIOR_TABLES["site"])
hour_keys, hour_vals = serialize_prior_table(PRIOR_TABLES["hour"])
month_keys, month_vals = serialize_prior_table(PRIOR_TABLES["month"])
site_hour_keys, site_hour_vals = serialize_prior_table(PRIOR_TABLES["site_hour"])

artifact = {
    "primary_labels": PRIMARY_LABELS,

    # prior
    "global_prob": PRIOR_TABLES["global"].astype(np.float32),
    "site_keys": site_keys,
    "site_vals": site_vals.astype(np.float32),
    "hour_keys": hour_keys,
    "hour_vals": hour_vals.astype(np.float32),
    "month_keys": month_keys,
    "month_vals": month_vals.astype(np.float32),
    "site_hour_keys": site_hour_keys,
    "site_hour_vals": site_hour_vals.astype(np.float32),

    # probe
    "scaler_mean": scaler_probe.mean_.astype(np.float32),
    "scaler_scale": scaler_probe.scale_.astype(np.float32),
    "pca_mean": pca_probe.mean_.astype(np.float32),
    "pca_components": pca_probe.components_.astype(np.float32),
    "active_class_idx": active_class_idx.astype(np.int32),
    "coef_mat": coef_mat.astype(np.float32),
    "intercept_vec": intercept_vec.astype(np.float32),

    # temporal-lite
    "temp_active_class_idx": temp_active_class_idx.astype(np.int32),
    "temp_coef_mat": temp_coef_mat.astype(np.float32),
    "temp_intercept_vec": temp_intercept_vec.astype(np.float32),
}

with open("offline_artifacts_v4_temporal.pkl", "wb") as f:
    pickle.dump(artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved offline_artifacts_v4_temporal.pkl")
print("File exists:", Path("offline_artifacts_v4_temporal.pkl").exists())
print("File size (MB):", Path("offline_artifacts_v4_temporal.pkl").stat().st_size / 1024 / 1024)

Saved offline_artifacts_v4_temporal.pkl
File exists: True
File size (MB): 0.8541507720947266


In [11]:
# A8 — Quick verification
with open("offline_artifacts_v4_temporal.pkl", "rb") as f:
    chk = pickle.load(f)

for k in [
    "global_prob", "site_vals", "hour_vals", "month_vals", "site_hour_vals",
    "scaler_mean", "scaler_scale", "pca_mean", "pca_components",
    "active_class_idx", "coef_mat", "intercept_vec",
    "temp_active_class_idx", "temp_coef_mat", "temp_intercept_vec",
]:
    v = chk[k]
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

global_prob (234,) float32
site_vals (8, 234) float32
hour_vals (13, 234) float32
month_vals (7, 234) float32
site_hour_vals (23, 234) float32
scaler_mean (1536,) float32
scaler_scale (1536,) float32
pca_mean (1536,) float32
pca_components (128, 1536) float32
active_class_idx (71,) int32
coef_mat (71, 128) float32
intercept_vec (71,) float32
temp_active_class_idx (71,) int32
temp_coef_mat (71, 4) float32
temp_intercept_vec (71,) float32


In [12]:
# A9 — Build teacher outputs on trusted full training windows
# This version is aligned with the V4 temporal-lite head trained earlier:
# temporal features = [mean, max, std, last-first]
# temporal params    = temp_coef_mat / temp_intercept_vec

ALPHA_PERCH = 0.60
ALPHA_PRIOR = 0.15
ALPHA_PROBE = 0.10
ALPHA_TEMP = 0.15

EPS = 1e-4

def logit_clip(p, eps=EPS):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

# -------------------------------------------------
# 1) build window-level metadata aligned with full_truth
# -------------------------------------------------
meta_full_teacher = full_truth[["row_id", "filename", "site", "hour_utc", "month"]].copy().reset_index(drop=True)

# -------------------------------------------------
# 2) prior logits on full_truth windows
# -------------------------------------------------
def build_prior_logits_from_meta(meta_df, prior_tables):
    n = len(meta_df)
    prior_prob = np.zeros((n, N_CLASSES), dtype=np.float32)

    global_prob = prior_tables["global"]

    for i, row in meta_df.reset_index(drop=True).iterrows():
        p = global_prob.copy()

        site = row["site"]
        hour = int(row["hour_utc"])
        month = int(row["month"])

        vecs = []

        if site in prior_tables["site"].index:
            vecs.append(prior_tables["site"].loc[site].to_numpy(dtype=np.float32))
        if hour in prior_tables["hour"].index:
            vecs.append(prior_tables["hour"].loc[hour].to_numpy(dtype=np.float32))
        if month in prior_tables["month"].index:
            vecs.append(prior_tables["month"].loc[month].to_numpy(dtype=np.float32))
        if (site, hour) in prior_tables["site_hour"].index:
            vecs.append(prior_tables["site_hour"].loc[(site, hour)].to_numpy(dtype=np.float32))

        if len(vecs) > 0:
            p = np.vstack([p] + vecs).mean(axis=0).astype(np.float32)

        prior_prob[i] = p

    return logit_clip(prior_prob)

prior_full_logits = build_prior_logits_from_meta(meta_full_teacher, PRIOR_TABLES)

# -------------------------------------------------
# 3) probe logits on full_truth windows
# emb_full_probe already corresponds to the trusted full windows
# -------------------------------------------------
X_full_scaled = (emb_full_probe.astype(np.float32) - scaler_probe.mean_[None, :]) / scaler_probe.scale_[None, :]
X_full_centered = X_full_scaled - pca_probe.mean_[None, :]
X_full_pca = X_full_centered @ pca_probe.components_.T

probe_full_logits = np.zeros((len(meta_full_teacher), N_CLASSES), dtype=np.float32)
probe_full_logits[:, active_class_idx] = (
    X_full_pca @ coef_mat.T + intercept_vec[None, :]
).astype(np.float32)

# -------------------------------------------------
# 4) temporal-lite logits on full_truth windows
# scores_full_raw_probe shape should be (n_full_files * 12, 234)
# temporal-lite features = [mean, max, std, last-first]
# -------------------------------------------------
n_full_files = len(full_files)

assert scores_full_raw_probe.shape[0] == n_full_files * N_WINDOWS, (
    f"scores_full_raw_probe has shape {scores_full_raw_probe.shape}, "
    f"expected first dim = {n_full_files * N_WINDOWS}"
)

seq_scores = scores_full_raw_probe.reshape(n_full_files, N_WINDOWS, N_CLASSES)

feat_mean = seq_scores.mean(axis=1)                         # (n_files, 234)
feat_max = seq_scores.max(axis=1)                          # (n_files, 234)
feat_std = seq_scores.std(axis=1)                          # (n_files, 234)
feat_delta = seq_scores[:, -1, :] - seq_scores[:, 0, :]   # (n_files, 234)

temp_file_logits = np.zeros((n_full_files, N_CLASSES), dtype=np.float32)

for k, j in enumerate(temp_active_class_idx):
    Xj = np.stack([
        feat_mean[:, j],
        feat_max[:, j],
        feat_std[:, j],
        feat_delta[:, j],
    ], axis=1).astype(np.float32)

    temp_file_logits[:, j] = (
        Xj @ temp_coef_mat[k].reshape(4, 1)
    ).reshape(-1).astype(np.float32) + temp_intercept_vec[k]

# expand file-level temporal logits back to 12 windows per file
temp_full_logits = np.repeat(temp_file_logits, N_WINDOWS, axis=0)

# -------------------------------------------------
# 5) fused teacher logits / probs
# -------------------------------------------------
perch_full_logits = scores_full_raw_probe.astype(np.float32)

teacher_full_logits = (
    ALPHA_PERCH * perch_full_logits
    + ALPHA_PRIOR * prior_full_logits
    + ALPHA_PROBE * probe_full_logits
    + ALPHA_TEMP * temp_full_logits
).astype(np.float32)

teacher_full_probs = 1.0 / (1.0 + np.exp(-np.clip(teacher_full_logits, -30, 30)))

# -------------------------------------------------
# 6) diagnostics
# -------------------------------------------------
print("meta_full_teacher:", meta_full_teacher.shape)
print("perch_full_logits:", perch_full_logits.shape)
print("prior_full_logits:", prior_full_logits.shape)
print("probe_full_logits:", probe_full_logits.shape)
print("temp_full_logits:", temp_full_logits.shape)
print("teacher_full_logits:", teacher_full_logits.shape)
print("teacher_full_probs:", teacher_full_probs.shape)
print("teacher probs range:", float(teacher_full_probs.min()), "to", float(teacher_full_probs.max()))

meta_full_teacher: (708, 5)
perch_full_logits: (708, 234)
prior_full_logits: (708, 234)
probe_full_logits: (708, 234)
temp_full_logits: (708, 234)
teacher_full_logits: (708, 234)
teacher_full_probs: (708, 234)
teacher probs range: 2.4437607862637378e-05 to 0.9999984502792358


In [13]:
# A10 — Build file-level teacher summaries

teacher_file_probs_max = teacher_full_probs.reshape(n_full_files, N_WINDOWS, N_CLASSES).max(axis=1).astype(np.float32)
teacher_file_probs_mean = teacher_full_probs.reshape(n_full_files, N_WINDOWS, N_CLASSES).mean(axis=1).astype(np.float32)

teacher_file_meta = pd.DataFrame({
    "filename": full_files,
})

print("teacher_file_probs_max:", teacher_file_probs_max.shape)
print("teacher_file_probs_mean:", teacher_file_probs_mean.shape)
print("teacher_file_meta:", teacher_file_meta.shape)

teacher_file_probs_max: (59, 234)
teacher_file_probs_mean: (59, 234)
teacher_file_meta: (59, 1)


In [14]:
# A11 — Save teacher cache for V5.0

teacher_cache = {
    "primary_labels": PRIMARY_LABELS,

    # window-level meta
    "meta_full_teacher": meta_full_teacher.copy(),

    # window-level components
    "perch_full_logits": perch_full_logits.astype(np.float32),
    "prior_full_logits": prior_full_logits.astype(np.float32),
    "probe_full_logits": probe_full_logits.astype(np.float32),
    "temp_full_logits": temp_full_logits.astype(np.float32),

    # window-level fused outputs
    "teacher_full_logits": teacher_full_logits.astype(np.float32),
    "teacher_full_probs": teacher_full_probs.astype(np.float32),

    # file-level meta / outputs
    "teacher_file_meta": teacher_file_meta.copy(),
    "teacher_file_probs_max": teacher_file_probs_max.astype(np.float32),
    "teacher_file_probs_mean": teacher_file_probs_mean.astype(np.float32),

    # optional ground truth for analysis
    "Y_FULL_TRUTH": Y_FULL_TRUTH.astype(np.uint8),
    "Y_FILE_TRUTH": Y_FILE_TRUTH.astype(np.uint8),
}

with open("teacher_cache_v5_0.pkl", "wb") as f:
    pickle.dump(teacher_cache, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved teacher_cache_v5_0.pkl")
print("File exists:", Path("teacher_cache_v5_0.pkl").exists())
print("File size (MB):", Path("teacher_cache_v5_0.pkl").stat().st_size / 1024 / 1024)

Saved teacher_cache_v5_0.pkl
File exists: True
File size (MB): 4.122065544128418


In [15]:
# A12 — Teacher diagnostics summary

primary_labels = PRIMARY_LABELS
teacher_window_probs = teacher_full_probs
teacher_file_probs_max_local = teacher_file_probs_max
teacher_file_probs_mean_local = teacher_file_probs_mean

print("Window-level teacher probs:")
print("  shape:", teacher_window_probs.shape)
print("  min :", float(np.min(teacher_window_probs)))
print("  mean:", float(np.mean(teacher_window_probs)))
print("  max :", float(np.max(teacher_window_probs)))
print()

print("File-level teacher probs (max pooling):")
print("  shape:", teacher_file_probs_max_local.shape)
print("  min :", float(np.min(teacher_file_probs_max_local)))
print("  mean:", float(np.mean(teacher_file_probs_max_local)))
print("  max :", float(np.max(teacher_file_probs_max_local)))
print()

for thr in [0.90, 0.95, 0.98, 0.99]:
    n_win = int((teacher_window_probs >= thr).sum())
    n_file = int((teacher_file_probs_max_local >= thr).sum())
    print(f"Threshold >= {thr:.2f}: window positives = {n_win}, file positives = {n_file}")
print()

# Per-class confident counts
thr = 0.95
win_counts = (teacher_window_probs >= thr).sum(axis=0)
file_counts = (teacher_file_probs_max_local >= thr).sum(axis=0)

top_win_idx = np.argsort(-win_counts)[:20]
top_file_idx = np.argsort(-file_counts)[:20]

print(f"Top 20 classes by window-level count >= {thr}:")
for idx in top_win_idx:
    if win_counts[idx] > 0:
        print(f"  {primary_labels[idx]}: {int(win_counts[idx])}")
print()

print(f"Top 20 classes by file-level count >= {thr}:")
for idx in top_file_idx:
    if file_counts[idx] > 0:
        print(f"  {primary_labels[idx]}: {int(file_counts[idx])}")
print()

# File-level quick compare
file_max_peak = teacher_file_probs_max_local.max(axis=1)
file_mean_peak = teacher_file_probs_mean_local.max(axis=1)

teacher_file_diag = pd.DataFrame({
    "filename": teacher_file_meta["filename"].values,
    "file_max_peak": file_max_peak,
    "file_mean_peak": file_mean_peak,
    "top_label_by_max": [primary_labels[i] for i in teacher_file_probs_max_local.argmax(axis=1)],
    "top_label_by_mean": [primary_labels[i] for i in teacher_file_probs_mean_local.argmax(axis=1)],
})

print("Top files by max confidence:")
display(teacher_file_diag.sort_values("file_max_peak", ascending=False).head(20))

Window-level teacher probs:
  shape: (708, 234)
  min : 2.4437607862637378e-05
  mean: 0.3273831009864807
  max : 0.9999984502792358

File-level teacher probs (max pooling):
  shape: (59, 234)
  min : 0.00048402528045699
  mean: 0.4389844238758087
  max : 0.9999984502792358

Threshold >= 0.90: window positives = 2505, file positives = 478
Threshold >= 0.95: window positives = 1627, file positives = 263
Threshold >= 0.98: window positives = 1162, file positives = 164
Threshold >= 0.99: window positives = 966, file positives = 136

Top 20 classes by window-level count >= 0.95:
  65380: 274
  24279: 154
  555146: 153
  22973: 146
  23158: 119
  66971: 92
  whtdov: 72
  compau: 54
  chacha1: 50
  47158son07: 48
  326272: 44
  undtin1: 39
  chvcon1: 35
  22967: 35
  47158son25: 27
  trsowl: 24
  litnig1: 21
  compot1: 15
  fepowl: 15
  orwpar: 14

Top 20 classes by file-level count >= 0.95:
  65380: 29
  24279: 19
  555146: 16
  23158: 15
  22973: 15
  66971: 11
  undtin1: 11
  compau: 10
 

,filename,file_max_peak,file_mean_peak,top_label_by_max,top_label_by_mean
25,BC2026_Train_0033_S22_20211216_200000.ogg,0.999998,0.999992,23158,23158
51,BC2026_Train_0059_S15_20250617_060200.ogg,0.999998,0.999928,whtdov,whtdov
49,BC2026_Train_0057_S15_20250617_060000.ogg,0.999997,0.999931,whtdov,whtdov
26,BC2026_Train_0034_S22_20211222_013000.ogg,0.999997,0.999988,23158,23158
50,BC2026_Train_0058_S15_20250617_060100.ogg,0.999996,0.999932,whtdov,whtdov
20,BC2026_Train_0028_S22_20211129_190000.ogg,0.999994,0.999976,23158,23158
29,BC2026_Train_0037_S22_20211226_021500.ogg,0.999989,0.994045,24279,23158
52,BC2026_Train_0060_S15_20250617_062700.ogg,0.999988,0.999865,whtdov,whtdov
32,BC2026_Train_0040_S22_20220101_014500.ogg,0.999983,0.995701,23158,23158
58,BC2026_Train_0066_S23_20241124_044002.ogg,0.999979,0.996131,chacha1,chacha1


In [16]:
# A13 — Pseudo-label threshold analysis

threshold_grid = [
    {"name": "very_strict", "pos_thr": 0.99, "neg_thr": 0.01, "file_thr": 0.99},
    {"name": "strict",      "pos_thr": 0.98, "neg_thr": 0.02, "file_thr": 0.98},
    {"name": "balanced",    "pos_thr": 0.95, "neg_thr": 0.05, "file_thr": 0.95},
]

analysis_rows = []

for cfg_thr in threshold_grid:
    name = cfg_thr["name"]
    pos_thr = cfg_thr["pos_thr"]
    neg_thr = cfg_thr["neg_thr"]
    file_thr = cfg_thr["file_thr"]

    # file-level gating: only allow window positives if file max also strong
    file_pos_mask = (teacher_file_probs_max >= file_thr)  # (n_files, n_classes)
    file_pos_mask_expanded = np.repeat(file_pos_mask, N_WINDOWS, axis=0)  # (708, 234)

    pseudo_pos = (teacher_full_probs >= pos_thr) & file_pos_mask_expanded
    pseudo_neg = (teacher_full_probs <= neg_thr)
    pseudo_keep = pseudo_pos | pseudo_neg

    pos_count = int(pseudo_pos.sum())
    neg_count = int(pseudo_neg.sum())
    keep_count = int(pseudo_keep.sum())

    pos_class_coverage = int((pseudo_pos.sum(axis=0) > 0).sum())
    neg_class_coverage = int((pseudo_neg.sum(axis=0) > 0).sum())

    analysis_rows.append({
        "name": name,
        "pos_thr": pos_thr,
        "neg_thr": neg_thr,
        "file_thr": file_thr,
        "pseudo_pos_count": pos_count,
        "pseudo_neg_count": neg_count,
        "pseudo_keep_count": keep_count,
        "pos_class_coverage": pos_class_coverage,
        "neg_class_coverage": neg_class_coverage,
    })

pseudo_label_analysis_df = pd.DataFrame(analysis_rows)
display(pseudo_label_analysis_df)

pseudo_label_analysis_df.to_csv("pseudo_label_analysis.csv", index=False)
print("Saved pseudo_label_analysis.csv")# A14 — Export pseudo labels (default: strict setting)

POS_THR = 0.98
NEG_THR = 0.02
FILE_THR = 0.98

file_pos_mask = (teacher_file_probs_max >= FILE_THR)
file_pos_mask_expanded = np.repeat(file_pos_mask, N_WINDOWS, axis=0)

pseudo_pos_mask = (teacher_full_probs >= POS_THR) & file_pos_mask_expanded
pseudo_neg_mask = (teacher_full_probs <= NEG_THR)
pseudo_keep_mask = pseudo_pos_mask | pseudo_neg_mask

# hard labels:
# 1 = pseudo positive
# 0 = pseudo negative
# -1 = ignore
pseudo_hard_labels = np.full_like(teacher_full_probs, fill_value=-1, dtype=np.int8)
pseudo_hard_labels[pseudo_neg_mask] = 0
pseudo_hard_labels[pseudo_pos_mask] = 1

# mask for student training: 1 means supervised, 0 means ignore
pseudo_train_mask = pseudo_keep_mask.astype(np.uint8)

print("pseudo_pos_count:", int((pseudo_hard_labels == 1).sum()))
print("pseudo_neg_count:", int((pseudo_hard_labels == 0).sum()))
print("pseudo_ignore_count:", int((pseudo_hard_labels == -1).sum()))

# window-level export table (long format, only supervised entries)
row_idx, cls_idx = np.where(pseudo_train_mask == 1)

pseudo_labels_window_df = pd.DataFrame({
    "row_id": meta_full_teacher.iloc[row_idx]["row_id"].values,
    "filename": meta_full_teacher.iloc[row_idx]["filename"].values,
    "site": meta_full_teacher.iloc[row_idx]["site"].values,
    "hour_utc": meta_full_teacher.iloc[row_idx]["hour_utc"].values,
    "month": meta_full_teacher.iloc[row_idx]["month"].values,
    "class_idx": cls_idx.astype(np.int32),
    "primary_label": [PRIMARY_LABELS[j] for j in cls_idx],
    "pseudo_label": pseudo_hard_labels[row_idx, cls_idx].astype(np.int8),
    "teacher_prob": teacher_full_probs[row_idx, cls_idx].astype(np.float32),
})

display(pseudo_labels_window_df.head(20))

pseudo_labels_window_df.to_parquet("pseudo_labels_window.parquet", index=False)

# file-level export table
file_row_idx, file_cls_idx = np.where(file_pos_mask == 1)
pseudo_labels_file_df = pd.DataFrame({
    "filename": teacher_file_meta.iloc[file_row_idx]["filename"].values,
    "class_idx": file_cls_idx.astype(np.int32),
    "primary_label": [PRIMARY_LABELS[j] for j in file_cls_idx],
    "file_prob_max": teacher_file_probs_max[file_row_idx, file_cls_idx].astype(np.float32),
    "file_prob_mean": teacher_file_probs_mean[file_row_idx, file_cls_idx].astype(np.float32),
})

pseudo_labels_file_df.to_parquet("pseudo_labels_file.parquet", index=False)

print("Saved pseudo_labels_window.parquet")
print("Saved pseudo_labels_file.parquet")

,name,pos_thr,neg_thr,file_thr,pseudo_pos_count,pseudo_neg_count,pseudo_keep_count,pos_class_coverage,neg_class_coverage
0,very_strict,0.99,0.01,0.99,966,2384,3350,30,8
1,strict,0.98,0.02,0.98,1162,3562,4724,38,12
2,balanced,0.95,0.05,0.95,1627,7406,9033,67,31


Saved pseudo_label_analysis.csv
pseudo_pos_count: 1162
pseudo_neg_count: 3562
pseudo_ignore_count: 160948


,row_id,filename,site,hour_utc,month,class_idx,primary_label,pseudo_label,teacher_prob
0,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,4,1595929,0,0.014628
1,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,9,22967,0,0.017946
2,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,13,23150,0,0.003029
3,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,14,23154,0,0.008227
4,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,21,24321,0,0.010374
5,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,55,476521,0,0.005208
6,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,57,517063,0,0.006514
7,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,4,1595929,0,0.012237
8,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,9,22967,0,0.017390
9,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,13,23150,0,0.003186


Saved pseudo_labels_window.parquet
Saved pseudo_labels_file.parquet


In [17]:
# A14 — Export pseudo labels (default: strict setting)

POS_THR = 0.98
NEG_THR = 0.02
FILE_THR = 0.98

file_pos_mask = (teacher_file_probs_max >= FILE_THR)
file_pos_mask_expanded = np.repeat(file_pos_mask, N_WINDOWS, axis=0)

pseudo_pos_mask = (teacher_full_probs >= POS_THR) & file_pos_mask_expanded
pseudo_neg_mask = (teacher_full_probs <= NEG_THR)
pseudo_keep_mask = pseudo_pos_mask | pseudo_neg_mask

# hard labels:
# 1 = pseudo positive
# 0 = pseudo negative
# -1 = ignore
pseudo_hard_labels = np.full_like(teacher_full_probs, fill_value=-1, dtype=np.int8)
pseudo_hard_labels[pseudo_neg_mask] = 0
pseudo_hard_labels[pseudo_pos_mask] = 1

# mask for student training: 1 means supervised, 0 means ignore
pseudo_train_mask = pseudo_keep_mask.astype(np.uint8)

print("pseudo_pos_count:", int((pseudo_hard_labels == 1).sum()))
print("pseudo_neg_count:", int((pseudo_hard_labels == 0).sum()))
print("pseudo_ignore_count:", int((pseudo_hard_labels == -1).sum()))

# window-level export table (long format, only supervised entries)
row_idx, cls_idx = np.where(pseudo_train_mask == 1)

pseudo_labels_window_df = pd.DataFrame({
    "row_id": meta_full_teacher.iloc[row_idx]["row_id"].values,
    "filename": meta_full_teacher.iloc[row_idx]["filename"].values,
    "site": meta_full_teacher.iloc[row_idx]["site"].values,
    "hour_utc": meta_full_teacher.iloc[row_idx]["hour_utc"].values,
    "month": meta_full_teacher.iloc[row_idx]["month"].values,
    "class_idx": cls_idx.astype(np.int32),
    "primary_label": [PRIMARY_LABELS[j] for j in cls_idx],
    "pseudo_label": pseudo_hard_labels[row_idx, cls_idx].astype(np.int8),
    "teacher_prob": teacher_full_probs[row_idx, cls_idx].astype(np.float32),
})

display(pseudo_labels_window_df.head(20))

pseudo_labels_window_df.to_parquet("pseudo_labels_window.parquet", index=False)

# file-level export table
file_row_idx, file_cls_idx = np.where(file_pos_mask == 1)
pseudo_labels_file_df = pd.DataFrame({
    "filename": teacher_file_meta.iloc[file_row_idx]["filename"].values,
    "class_idx": file_cls_idx.astype(np.int32),
    "primary_label": [PRIMARY_LABELS[j] for j in file_cls_idx],
    "file_prob_max": teacher_file_probs_max[file_row_idx, file_cls_idx].astype(np.float32),
    "file_prob_mean": teacher_file_probs_mean[file_row_idx, file_cls_idx].astype(np.float32),
})

pseudo_labels_file_df.to_parquet("pseudo_labels_file.parquet", index=False)

print("Saved pseudo_labels_window.parquet")
print("Saved pseudo_labels_file.parquet")

pseudo_pos_count: 1162
pseudo_neg_count: 3562
pseudo_ignore_count: 160948


,row_id,filename,site,hour_utc,month,class_idx,primary_label,pseudo_label,teacher_prob
0,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,4,1595929,0,0.014628
1,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,9,22967,0,0.017946
2,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,13,23150,0,0.003029
3,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,14,23154,0,0.008227
4,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,21,24321,0,0.010374
5,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,55,476521,0,0.005208
6,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,57,517063,0,0.006514
7,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,4,1595929,0,0.012237
8,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,9,22967,0,0.017390
9,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,3,6,13,23150,0,0.003186


Saved pseudo_labels_window.parquet
Saved pseudo_labels_file.parquet


In [18]:
# A15 — Export student-ready arrays

# student-ready arrays aligned with meta_full_teacher row order
student_targets_window = pseudo_hard_labels.astype(np.int8)      # (708, 234), values in {-1,0,1}
student_mask_window = pseudo_train_mask.astype(np.uint8)         # (708, 234), values in {0,1}
student_teacher_probs_window = teacher_full_probs.astype(np.float32)

np.save("student_targets_window.npy", student_targets_window)
np.save("student_mask_window.npy", student_mask_window)
np.save("student_teacher_probs_window.npy", student_teacher_probs_window)

# also save aligned metadata
meta_full_teacher.to_parquet("student_meta_window.parquet", index=False)

print("Saved student_targets_window.npy", student_targets_window.shape, student_targets_window.dtype)
print("Saved student_mask_window.npy", student_mask_window.shape, student_mask_window.dtype)
print("Saved student_teacher_probs_window.npy", student_teacher_probs_window.shape, student_teacher_probs_window.dtype)
print("Saved student_meta_window.parquet", meta_full_teacher.shape)

np.save("emb_full_probe.npy", emb_full_probe.astype(np.float32))
print("Saved emb_full_probe.npy", emb_full_probe.shape, emb_full_probe.dtype)

Saved student_targets_window.npy (708, 234) int8
Saved student_mask_window.npy (708, 234) uint8
Saved student_teacher_probs_window.npy (708, 234) float32
Saved student_meta_window.parquet (708, 5)
Saved emb_full_probe.npy (708, 1536) float32
